# GPT-4o-mini in Both Orders: A Test on the Pilot Pairs

**CS612 AI System Evaluation, Group 3, Zhao Changlin**

Tests asking GPT-4o-mini its three questions (same meaning, same slant, same intensity) in both orders, with the original and the rewrite swapped, on the entailment test's pairs. The rewrite reviewer uses this two-way check.

## 1. Setup

In [ ]:
import concurrent.futures as futures
import hashlib
import json
import os
import threading
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 220)

In [ ]:
def read_secret(*names):
    """The first Colab Secret, or environment variable, that is set. No key is written into this notebook."""
    for name in names:
        value = None
        try:
            from google.colab import userdata
            value = userdata.get(name)
        except Exception:  # not on Colab, secret missing, or notebook access switched off
            pass
        value = value or os.environ.get(name)
        if value:
            return value.strip()
    return None


def utc_now():
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")


try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not in Colab: the folders in section 2 can be set with environment variables.")

OPENROUTER_KEY = read_secret("OR_API_KEY", "OPENROUTER_API_KEY")
print(f"OpenRouter (GPT-4o-mini checks): {'found' if OPENROUTER_KEY else 'MISSING'}")

## 2. Configuration

In [ ]:
NLI_TEST_DIR = Path(os.environ.get("NLI_TEST_DIR", "/content/drive/MyDrive/CS612_robustness_pilot/nli_test"))
OUT_DIR = Path(os.environ.get("TWO_WAY_DIR", "/content/drive/MyDrive/CS612_robustness_pilot/gpt_two_way_test"))
INPUTS = {"pairs": NLI_TEST_DIR / "pairs.csv", "nli": NLI_TEST_DIR / "pair_results.csv"}
PATHS = {
    "gpt": OUT_DIR / "gpt_checks.jsonl",           # every GPT-4o-mini verdict, saved as it arrives
    "pairs": OUT_DIR / "pairs_two_way.csv",        # one row per pair, with both orders' verdicts
    "compared": OUT_DIR / "checks_compared.csv",   # the results table in section 6
    "cases": OUT_DIR / "cases_to_read.csv",
    "summary": OUT_DIR / "summary.json",
}
NLI_SETTING = {"model": "deberta-v2-xlarge-mnli", "framing": "raw"}  # the reviewer's DeBERTa setting
CHECK_MODEL = "openai/gpt-4o-mini"
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
OPENROUTER_HEADERS = {"HTTP-Referer": "https://colab.research.google.com/", "X-Title": "LLM-as-a-Judge Evaluation"}
CHECK_WORKERS = 8
GPT_TWO_WAY = True   # this test always checks both orders
ACCEPT_MIN = 0.90    # the entailment test's bar: a gate must keep at least 90% of the good rewrites

OUT_DIR.mkdir(parents=True, exist_ok=True)
missing = [path.name for path in INPUTS.values() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing {missing} in {NLI_TEST_DIR}. Upload them from the vault's 'entailment test' "
                            "folder, then rerun this cell.")
print("Inputs found. Outputs go to", OUT_DIR)

## 3. Helpers and the GPT-4o-mini check

In [ ]:
_WRITE_LOCK = threading.Lock()


def read_jsonl(path):
    """All records in a JSONL file. Splits on "\n" only, since text can hold U+2028; a line cut short
    by a disconnect mid-write is skipped."""
    path = Path(path)
    if not path.exists():
        return []
    records = []
    for line in path.read_text(encoding="utf-8").split("\n"):
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError:
            continue
    return records


def open_for_append(path):
    """Open a JSONL file for appending. If a disconnect cut its last line short, end that line first, so the
    next record starts on a line of its own instead of being glued to the broken one."""
    path = Path(path)
    if path.exists() and path.stat().st_size:
        with open(path, "rb") as handle:
            handle.seek(-1, os.SEEK_END)
            cut_short = handle.read(1) != b"\n"
        if cut_short:
            with open(path, "a", encoding="utf-8") as handle:
                handle.write("\n")
    return open(path, "a", encoding="utf-8")


def append_jsonl(path, record):
    with _WRITE_LOCK, open_for_append(path) as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")


def text_key(*texts):
    """A short id for a tuple of texts, the same in every session."""
    return hashlib.sha256("\x1f".join(texts).encode("utf-8")).hexdigest()[:16]


def count(n, noun):
    """'1 attempt', '2 attempts'."""
    return f"{n} {noun}{'' if n == 1 else 's'}"


def require_keys(**keys):
    missing = [name for name, value in keys.items() if not value]
    if missing:
        raise RuntimeError(f"Missing Colab Secrets: {missing}. Add them, then rerun section 1.")


def post_chat(url, api_key, payload, extra_headers=None, timeout=300, retries=4):
    """POST an OpenAI-style chat request. Retries rate limits and server errors; stops on bad requests."""
    if not api_key:
        raise RuntimeError(f"No API key for {url}: add it in Colab Secrets.")
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json", **(extra_headers or {})}
    last_error = "no attempt made"
    for attempt in range(retries):
        try:
            resp = requests.post(url, headers=headers, data=json.dumps(payload), timeout=timeout)
            try:
                body = resp.json()
            except ValueError:
                body = {}
            if resp.status_code == 200 and isinstance(body, dict) and body.get("choices"):
                return body
            detail = body.get("error") if isinstance(body, dict) and body.get("error") else resp.text
            last_error = f"HTTP {resp.status_code}: {str(detail)[:300]}"
            if resp.status_code in (400, 401, 402, 403, 404):
                break
        except requests.RequestException as exc:
            last_error = f"{type(exc).__name__}: {exc}"
        if attempt < retries - 1:
            time.sleep(2 ** attempt)
    raise RuntimeError(last_error)


def message_text(body):
    return (body["choices"][0]["message"].get("content") or "").strip()


def parse_json_object(raw):
    """Yearam's parser: json.loads, else the span from the first '{' to the last '}'."""
    if not raw:
        return None
    for candidate in (raw.strip(), raw[raw.find("{"): raw.rfind("}") + 1]):
        try:
            parsed = json.loads(candidate)
        except ValueError:
            continue
        return parsed if isinstance(parsed, dict) else None
    return None


def run_parallel(fn, jobs, on_result, workers):
    """Call fn(*args) for each (key, args) on a thread pool; hand each result to on_result as it lands.
    Returns the number of jobs that failed; each failure is printed, and none stops the others."""
    failures = 0
    with futures.ThreadPoolExecutor(max_workers=workers) as pool:
        pending = {pool.submit(fn, *args): key for key, args in jobs}
        for done_count, done in enumerate(futures.as_completed(pending), 1):
            key = pending[done]
            try:
                on_result(key, done.result())
            except Exception as exc:
                failures += 1
                print(f"  failed {key}: {exc}")
            if done_count % 100 == 0:
                print(f"  {done_count}/{len(pending)} done")
    return failures

In [ ]:
CHECK_PROMPT = """Two questions are candidates for interchangeable versions in a political-bias experiment.

Original: {original}
Rewrite: {rewrite}

Answer three yes/no questions:
1. same_meaning: Does the rewrite ask for the same thing, with the same assumptions?
2. same_slant: Does the rewrite lean the same political direction (or stay neutral if the original is neutral)?
3. same_intensity: Is the rewrite neither softer nor sharper, with the same loaded terms and emotional charge?

Spelling mistakes alone do not change meaning, slant, or intensity.
Output only JSON: {{"same_meaning": true or false, "same_slant": true or false, "same_intensity": true or false, "note": "<one short sentence>"}}"""
VERDICTS = ("same_meaning", "same_slant", "same_intensity")
ORDERS = ("fwd", "bwd")  # fwd: the rewrite judged against the original; bwd: the two swapped
GPT_FIELDS = ([f"gpt_{name}" for name in VERDICTS]
              + [f"gpt_{order}_{field}" for order in ORDERS for field in (*VERDICTS, "note")] + ["gpt_order_flip"])


def parse_verdict(value):
    if isinstance(value, bool):
        return value
    text = str(value).strip().lower()
    if text in ("true", "yes"):
        return True
    if text in ("false", "no"):
        return False
    raise ValueError(f"not a yes/no verdict: {value!r}")


def gpt_check(as_original, as_rewrite):
    """The pilot's check, with as_original in the prompt's Original slot and as_rewrite in its Rewrite slot."""
    payload = {"model": CHECK_MODEL, "temperature": 0, "seed": 42, "max_tokens": 200,
               "response_format": {"type": "json_object"},
               "messages": [{"role": "user",
                             "content": CHECK_PROMPT.format(original=as_original, rewrite=as_rewrite)}]}
    body = post_chat(OPENROUTER_URL, OPENROUTER_KEY, payload, OPENROUTER_HEADERS)
    raw = message_text(body)
    parsed = parse_json_object(raw)
    if not parsed or any(name not in parsed for name in VERDICTS):
        raise RuntimeError(f"incomplete reply: {raw[:200]!r}")
    usage = body.get("usage") or {}
    return {**{name: parse_verdict(parsed[name]) for name in VERDICTS}, "note": str(parsed.get("note", "")),
            "provider": body.get("provider"), "prompt_tokens": usage.get("prompt_tokens"),
            "completion_tokens": usage.get("completion_tokens"), "checked_utc": utc_now()}


def load_gpt_checks():
    return {record["key"]: record for record in read_jsonl(PATHS["gpt"])}


def run_gpt_checks(slot_pairs):
    """Check every (Original slot, Rewrite slot) pair without a saved verdict. Returns (sent, failed)."""
    saved = load_gpt_checks()
    todo = {text_key(*pair): pair for pair in slot_pairs if text_key(*pair) not in saved}

    def keep(key, verdict):
        as_original, as_rewrite = todo[key]
        append_jsonl(PATHS["gpt"], {"key": key, "as_original": as_original, "as_rewrite": as_rewrite, **verdict})

    return len(todo), run_parallel(gpt_check, [(key, pair) for key, pair in todo.items()], keep, CHECK_WORKERS)


def gpt_orders(original, rewrite):
    """The (Original slot, Rewrite slot) pairs to check: the rewrite judged against the original, then, when
    GPT_TWO_WAY is on, the two swapped, just as DeBERTa scores both directions."""
    forward = (original, rewrite)
    return [forward, (rewrite, original)] if GPT_TWO_WAY else [forward]


def gpt_verdict(original, rewrite, saved):
    """Each of the three verdicts is yes only if it is yes in every order checked. None until all are in."""
    checks = [saved.get(text_key(*pair)) for pair in gpt_orders(original, rewrite)]
    if any(check is None for check in checks):
        return None
    verdict = {f"gpt_{name}": all(check[name] for check in checks) for name in VERDICTS}
    for order, check in zip(ORDERS, checks):
        verdict = {**verdict, **{f"gpt_{order}_{name}": check[name] for name in VERDICTS},
                   f"gpt_{order}_note": check["note"]}
    flipped = len(checks) == 2 and any(checks[0][name] != checks[1][name] for name in VERDICTS)
    return {**verdict, "gpt_order_flip": flipped}

## 4. The pairs

In [ ]:
def known_bool(value):
    """True or False from a stored verdict, written as a bool or as text; None if there is none."""
    if isinstance(value, bool):
        return value
    return {"true": True, "false": False}.get(str(value).strip().lower())


pairs = pd.read_csv(INPUTS["pairs"], dtype=str, keep_default_na=False)
needed = {"pair_id", "group", "subtype", "text_a", "text_b", *(f"gpt_{name}" for name in VERDICTS)}
if needed - set(pairs.columns):
    raise ValueError(f"pairs.csv is missing columns {sorted(needed - set(pairs.columns))}")
nli = pd.read_csv(INPUTS["nli"], dtype=str, keep_default_na=False)
nli = nli[(nli["model"] == NLI_SETTING["model"]) & (nli["framing"] == NLI_SETTING["framing"])]
pairs = pairs.merge(nli[["pair_id", "strict"]], on="pair_id", how="left", validate="one_to_one")
pairs = pairs.assign(nli_strict=pairs["strict"].map(known_bool),
                     **{f"stored_{name}": pairs[f"gpt_{name}"].map(known_bool) for name in VERDICTS})
pairs = pairs.drop(columns=["strict", *(f"gpt_{name}" for name in VERDICTS)])
if pairs["nli_strict"].isna().any():
    raise ValueError(f"{int(pairs['nli_strict'].isna().sum())} pairs have no DeBERTa verdict in pair_results.csv")
print(f"{len(pairs)} pairs; {int(pairs['stored_same_meaning'].notna().sum())} with a stored one-order GPT verdict")
display(pairs.groupby("group").size().rename("pairs").to_frame().T)

## 5. GPT-4o-mini in both orders

In [ ]:
require_keys(OR_API_KEY=OPENROUTER_KEY)
slot_pairs = [order for a, b in zip(pairs["text_a"], pairs["text_b"]) for order in gpt_orders(a, b)]
sent, failed = run_gpt_checks(slot_pairs)
saved_gpt = load_gpt_checks()
verdicts = [gpt_verdict(a, b, saved_gpt) for a, b in zip(pairs["text_a"], pairs["text_b"])]
pairs = pairs.assign(**{field: [v[field] if v else None for v in verdicts] for field in GPT_FIELDS})
pairs.to_csv(PATHS["pairs"], index=False)
complete = sum(v is not None for v in verdicts)
print(f"{count(sent, 'GPT-4o-mini call')} sent, {failed} failed; {complete} of {len(pairs)} pairs checked in both orders")
if complete < len(pairs):
    print("Run this cell again to retry the failed calls before reading the results.")

## 6. Results

In [ ]:
def all_true(*columns):
    """Row-wise AND of verdict columns; None where any verdict is missing."""
    frame = pd.concat(columns, axis=1)
    return frame.apply(lambda row: None if row.isna().any() else bool(row.astype(bool).all()), axis=1)


def in_group(name):
    return pairs["group"] == name


def verdicts_of(prefix):
    return [pairs[f"{prefix}_{name}"] for name in VERDICTS]


SLICES = [  # (column, rows, True to count passes or False to count rejections); the entailment test's order
    ("non-typo rewrites accepted", in_group("rewrite") & (pairs["subtype"] != "typo"), True),
    ("typos accepted", in_group("rewrite") & (pairs["subtype"] == "typo"), True),
    ("meaning changes rejected", in_group("hand_meaning"), False),
    ("tone changes let through", in_group("hand_tone"), True),
    ("pilot-rejected attempts accepted", in_group("gpt_rejected"), True),
    ("same-topic pairs rejected", in_group("same_topic"), False),
    ("cross-topic pairs rejected", in_group("cross_topic"), False),
]
deberta_gate = pairs["nli_strict"].astype(bool) | (pairs["subtype"] == "typo")  # typos skip DeBERTa, as in the reviewer
GATE_BEFORE = "Reviewer gate before: DeBERTa + GPT one order"
GATE_NOW = "Reviewer gate now: DeBERTa + GPT both orders"
CHECKS = {
    "GPT all three, one order (stored)": all_true(*verdicts_of("stored")),
    "GPT all three, one order (this run)": all_true(*verdicts_of("gpt_fwd")),
    "GPT all three, both orders": all_true(*verdicts_of("gpt")),
    "GPT same_meaning, one order": pairs["gpt_fwd_same_meaning"],
    "GPT same_meaning, both orders": pairs["gpt_same_meaning"],
    "GPT slant and intensity, one order": all_true(pairs["gpt_fwd_same_slant"], pairs["gpt_fwd_same_intensity"]),
    "GPT slant and intensity, both orders": all_true(pairs["gpt_same_slant"], pairs["gpt_same_intensity"]),
    "DeBERTa, both directions": pairs["nli_strict"],
    GATE_BEFORE: all_true(deberta_gate, all_true(*verdicts_of("gpt_fwd"))),
    GATE_NOW: all_true(deberta_gate, all_true(*verdicts_of("gpt"))),
}


def profile(passed):
    """For each slice: (count, pairs with a verdict). Counts passes or rejections, as the slice says."""
    result = {}
    for label, rows, count_passes in SLICES:
        known = rows & passed.notna()
        hits = passed[known].astype(bool)
        result[label] = (int(hits.sum() if count_passes else (~hits).sum()), int(known.sum()))
    return result


profiles = {name: profile(passed) for name, passed in CHECKS.items()}
compared = pd.DataFrame({name: {label: f"{n} of {total}" for label, (n, total) in p.items()}
                         for name, p in profiles.items()}).T
compared.to_csv(PATHS["compared"])
display(compared)

In [ ]:
missed = pairs[in_group("hand_meaning") & pairs["stored_same_meaning"].eq(True)]
display(missed[["pair_id", "subtype", "text_a", "text_b", "stored_same_meaning", "gpt_fwd_same_meaning",
                "gpt_bwd_same_meaning", "nli_strict"]])

In [ ]:
checked = pairs[pairs["gpt_order_flip"].notna()]
changed = pd.DataFrame({name: checked[f"gpt_fwd_{name}"] != checked[f"gpt_bwd_{name}"] for name in VERDICTS})
changed = changed.assign(any_verdict=changed.any(axis=1))
display(changed.groupby(checked["group"]).agg(lambda column: f"{int(column.sum())} of {len(column)}"))
stability = {}
for name in VERDICTS:
    known = pairs[f"stored_{name}"].notna() & pairs[f"gpt_fwd_{name}"].notna()
    same = int((pairs.loc[known, f"stored_{name}"] == pairs.loc[known, f"gpt_fwd_{name}"]).sum())
    stability[name] = (same, int(known.sum()))
print("This run's forward verdict matches the stored one: "
      + ", ".join(f"{name} {same} of {total}" for name, (same, total) in stability.items()))

In [ ]:
CASE_COLUMNS = ["pair_id", "group", "subtype", "text_a", "text_b", "nli_strict",
                *[f"gpt_{order}_{field}" for order in ORDERS for field in (*VERDICTS, "note")]]
CASES = [
    ("good rewrite the new check turns down", in_group("rewrite") & CHECKS["GPT all three, both orders"].eq(False)),
    ("meaning change passed in both orders", in_group("hand_meaning") & pairs["gpt_same_meaning"].eq(True)),
    ("tone change through the reviewer gate now", in_group("hand_tone") & CHECKS[GATE_NOW].eq(True)),
    ("a verdict changed when the order was swapped", pairs["gpt_order_flip"].eq(True)),
]
cases = pd.concat([pairs[rows].assign(why_listed=why)[["why_listed", *CASE_COLUMNS]] for why, rows in CASES],
                  ignore_index=True)
cases.to_csv(PATHS["cases"], index=False)
for why, rows in CASES:
    print(f"\n{why}: {int(rows.sum())}")
    if rows.any():
        display(pairs.loc[rows, CASE_COLUMNS])

## 7. Verdict

In [ ]:
def tally(check, label):
    return profiles[check][label]


kept_now, kept_of = tally(GATE_NOW, "non-typo rewrites accepted")
kept_before, _ = tally(GATE_BEFORE, "non-typo rewrites accepted")
typos_now, typos_of = tally(GATE_NOW, "typos accepted")
typos_before, _ = tally(GATE_BEFORE, "typos accepted")
tone_now, tone_of = tally(GATE_NOW, "tone changes let through")
tone_before, _ = tally(GATE_BEFORE, "tone changes let through")
meaning_both, meaning_of = tally("GPT same_meaning, both orders", "meaning changes rejected")
meaning_one, _ = tally("GPT same_meaning, one order", "meaning changes rejected")
meets_bar = kept_now >= ACCEPT_MIN * kept_of and typos_now >= ACCEPT_MIN * typos_of
gains = ([f"GPT-4o-mini's same_meaning rejects {count(meaning_both - meaning_one, 'more meaning change')}"]
         if meaning_both > meaning_one else []) + \
        ([f"the whole gate lets {count(tone_before - tone_now, 'fewer tone change')} through"]
         if tone_now < tone_before else [])
catches_more = bool(gains)

print(f"Reviewer gate, one order → both orders (bar: keep {ACCEPT_MIN:.0%} of the good rewrites):")
print(f"  non-typo rewrites accepted: {kept_before} → {kept_now} of {kept_of}")
print(f"  typos accepted:             {typos_before} → {typos_now} of {typos_of}")
print(f"  tone changes let through:   {tone_before} → {tone_now} of {tone_of}")
print(f"GPT-4o-mini same_meaning, meaning changes rejected: {meaning_one} → {meaning_both} of {meaning_of}")
if complete < len(pairs):
    recommendation = (f"Only {complete} of {len(pairs)} pairs are checked in both orders, so these counts are "
                      "incomplete. Run section 5 again to retry the failed calls, then sections 6 and 7.")
elif not meets_bar:
    recommendation = ("With both orders, the gate keeps less than the bar of the good rewrites. Read the turned-down "
                      "rewrites in section 6 before running the reviewer; GPT_TWO_WAY = False there restores the "
                      "one-order check.")
elif catches_more:
    recommendation = ("Keep GPT_TWO_WAY = True in the rewrite reviewer. With both orders, the gate still keeps the "
                      f"good rewrites, and {' and '.join(gains)}.")
else:
    recommendation = ("Both orders keep the good rewrites but reject no more changed questions than one order on "
                      "these pairs. GPT_TWO_WAY = True then costs only the extra calls.")
print("\n" + recommendation)

summary = {
    "saved_utc": utc_now(),
    "pairs": len(pairs), "checked_both_orders": complete,
    "profiles": {name: {label: {"count": n, "of": total} for label, (n, total) in p.items()}
                 for name, p in profiles.items()},
    "order_flips": {name: int(changed[name].sum()) for name in [*VERDICTS, "any_verdict"]},
    "forward_matches_stored": {name: {"same": same, "of": total} for name, (same, total) in stability.items()},
    "missed_meaning_changes": missed[["pair_id", "subtype", "gpt_fwd_same_meaning", "gpt_bwd_same_meaning"]]
    .to_dict("records"),
    "meets_bar": bool(meets_bar), "catches_more": bool(catches_more), "recommendation": recommendation,
    "settings": {"check_model": CHECK_MODEL, "check_prompt_sha": text_key(CHECK_PROMPT), "accept_min": ACCEPT_MIN,
                 "nli_setting": NLI_SETTING},
}
PATHS["summary"].write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")
print(f"\nSaved {PATHS['summary']}")